# Medication dosing vs physician age

In [ ]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/rega_data/trauma_categories_Rega Pain Study15.09.2025_v2.xlsx'
medic_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/rega_data/rega_physician_list_09102025.xlsx'
meta_medic_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/data/medreg_extraction/joined_final_complete_extractions_20251008_221735.xlsx'

In [ ]:
restrict_to_trauma = True
restrict_to_primary = True

In [ ]:
data_df = pd.read_excel(data_path)
medic_df = pd.read_excel(medic_data_path)
meta_medic_df = pd.read_excel(meta_medic_data_path)

In [ ]:
medic_df['full_name'] = medic_df['Mitglieder mit Einsatzfunktion'].str.replace(' (Flugarzt/Flugärztin)', '')
medic_df.drop_duplicates(subset=['Mitglieder mit Einsatzfunktion'], inplace=True)
medic_df = medic_df.merge(meta_medic_df, how='left', on='full_name')
medic_df.rename(columns={'Sex m/w': 'physician_sex'}, inplace=True)
data_df = data_df.merge(medic_df, how='left', left_on='Mitglieder mit Einsatzfunktion', right_on='Mitglieder mit Einsatzfunktion')

In [ ]:
duplicates = data_df[data_df["SNZ Ereignis Nr. "].duplicated()]["SNZ Ereignis Nr. "]
print(f'Duplicates found: {duplicates.nunique()}')
# drop duplicates
data_df = data_df.drop_duplicates(subset=["SNZ Ereignis Nr. "])

In [ ]:
n_vas_under4 = data_df[data_df["VAS_on_scene"] <= 3].shape[0]
print(f'Excluded {n_vas_under4} patients with VAS <= 3')

# adult patients with vas <= 3
n_adult_vas_under4 = data_df[(data_df["VAS_on_scene"] <= 3) & (data_df["Alter "] >= 16)].shape[0]
print(f'Excluded {n_adult_vas_under4} adult patients with VAS <= 3')

# pediatric patients with vas <= 3
n_pediatric_vas_under4 = data_df[(data_df["VAS_on_scene"] <= 3) & (data_df["Alter "] < 16)].shape[0]
print(f'Excluded {n_pediatric_vas_under4} pediatric patients with VAS <= 3')

data_df = data_df[data_df["VAS_on_scene"] > 3]

In [ ]:
if restrict_to_trauma:
    n_non_trauma = data_df[data_df['Einteilung (reduziert)'] != 'Unfall'].shape[0]
    print(f'Excluded {n_non_trauma} non-trauma patients')

    # adult non-trauma patients
    n_adult_non_trauma = data_df[(data_df['Einteilung (reduziert)'] != 'Unfall') & (data_df["Alter "] >= 16)].shape[0]
    print(f'Excluded {n_adult_non_trauma} adult non-trauma patients')
    # pediatric non-trauma patients
    n_pediatric_non_trauma = data_df[(data_df['Einteilung (reduziert)'] != 'Unfall') & (data_df["Alter "] < 16)].shape[0]
    print(f'Excluded {n_pediatric_non_trauma} pediatric non-trauma patients')

    data_df = data_df[data_df['Einteilung (reduziert)'] == 'Unfall']

In [ ]:
if restrict_to_primary:
    n_secondary = data_df[data_df['Einsatzart'] != 'Primär'].shape[0]
    print(f'Excluded {n_secondary} secondary transport patients')

    # adult secondary transport patients
    n_adult_secondary = data_df[(data_df['Einsatzart'] != 'Primär') & (data_df["Alter "] >= 16)].shape[0]
    print(f'Excluded {n_adult_secondary} adult secondary transport patients')
    # pediatric secondary transport patients
    n_pediatric_secondary = data_df[(data_df['Einsatzart'] != 'Primär') & (data_df["Alter "] < 16)].shape[0]
    print(f'Excluded {n_pediatric_secondary} pediatric secondary transport patients')
    data_df = data_df[data_df['Einsatzart'] == 'Primär']


In [ ]:
adult_df = data_df[data_df["Alter "] >= 16]
pediatric_df = data_df[data_df["Alter "] < 16]

In [ ]:
adult_df = adult_df[~adult_df['VAS_on_arrival'].isna()]

In [ ]:
len(adult_df)

In [ ]:
adult_df['event_year'] = pd.to_datetime(adult_df['Ereignisdatum'], format='%d.%m.%Y').dt.year
adult_df['physician_age'] = adult_df['event_year'] - adult_df['year_of_birth']
# physician year of final exam (from licence_date which can be either d.m.Y or Y)
adult_df['physician_licence_year'] = adult_df['licence_date'].apply(lambda x: str(x).split('.')[-1] if '.' in str(x) else str(x))
adult_df['phyisician_experience_years'] = adult_df['event_year'] - pd.to_numeric(adult_df['physician_licence_year'], errors='coerce')


In [ ]:
# Create medication dose variables (matching Table 1 approach)
adult_df['fentanyl_dose'] = 0
adult_df['ketamine_dose'] = 0
adult_df['esketamine_dose'] = 0
adult_df['morphine_dose'] = 0
adult_df['Alle Medikamente'] = adult_df['Alle Medikamente'].str.replace(',', ';')  # replace commas with semicolons for consistency
for i, row in adult_df.iterrows():
    if pd.isna(row['Alle Medikamente']) or row['Alle Medikamente'] == 0:
        continue
    for analgetic in row['Alle Medikamente'].split(';'):
        if analgetic.strip() == '':
            continue
        # remove mcg or mg from dose
        if '7IE' in analgetic:
                print(f"Skipping dose with 7IE: {analgetic}")
                continue

        analgetic = analgetic.replace('mcg', '').replace('mg', '').strip()
        if 'Fentanyl' in analgetic and '/h' not in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip()
            adult_df.at[i, 'fentanyl_dose'] += float(dose) 
        elif 'Fentanyl' in analgetic and '/h' in analgetic:
            dose = analgetic.split('Fentanyl')[-1].strip().replace('/h', '')
            dose = float(dose) * adult_df.at[i, 'mission_duration']  
            adult_df.at[i, 'fentanyl_dose'] += float(dose)
        elif 'Ketamin' in analgetic or 'Ketamine' in analgetic:
            dose = analgetic.split('Ketamin')[-1].strip()
            adult_df.at[i, 'ketamine_dose'] += float(dose)
        elif 'Esketamin' in analgetic:
            dose = analgetic.split('Esketamin')[-1].strip()
            adult_df.at[i, 'esketamine_dose'] += float(dose)
        elif 'Morphin' in analgetic or 'Morphine' in analgetic:
            dose = analgetic.split('Morphin')[-1].strip()
            adult_df.at[i, 'morphine_dose'] += float(dose)

# Create medication variables
adult_df['fentanyl_given'] = adult_df['fentanyl_dose'] > 0
adult_df['morphine_given'] = adult_df['morphine_dose'] > 0
adult_df['ketamine_given'] = adult_df['ketamine_dose'] > 0
adult_df['esketamine_given'] = adult_df['esketamine_dose'] > 0

# Create combined medication variables (PRIMARY VARIABLES OF INTEREST)
adult_df['any_opiate_dose'] = adult_df['morphine_dose'] + adult_df['fentanyl_dose']
adult_df['any_ketamine_dose'] = adult_df['ketamine_dose'] + adult_df['esketamine_dose']
adult_df['any_opiate_given'] = (adult_df['morphine_dose'] > 0) | (adult_df['fentanyl_dose'] > 0)
adult_df['any_ketamine_given'] = (adult_df['ketamine_dose'] > 0) | (adult_df['esketamine_dose'] > 0)



In [ ]:
# Publication-ready plot: physician age vs medication dose (including zero doses)
import os

dose_columns = ["any_opiate_dose", "any_ketamine_dose"]
metric_columns = [
    ("physician_age", "Physician age (years)"),
    ("phyisician_experience_years", "Physician experience (years)"),
]

def plot_dose_vs_physician_metric(
    df,
    x_col,
    x_label,
    include_zero_dose=True,
    show_main_title=True,
    save_path=None,
):
    plot_df = df[[x_col] + dose_columns].copy()
    plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[x_col])
    plot_df = plot_df.melt(
        id_vars=[x_col],
        value_vars=dose_columns,
        var_name="medication_group",
        value_name="dose",
    ).dropna(subset=["dose"])

    if include_zero_dose:
        plot_df = plot_df[plot_df["dose"] >= 0]
        zero_text = "Including zero doses"
    else:
        plot_df = plot_df[plot_df["dose"] > 0]
        zero_text = "Excluding zero doses"

    if plot_df.empty:
        print(f"No data available for {x_col} ({zero_text.lower()}).")
        return

    sns.set_theme(style="whitegrid", context="paper")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), dpi=160)
    group_order = ["any_opiate_dose", "any_ketamine_dose"]
    group_labels = {
        "any_opiate_dose": "Opiate",
        "any_ketamine_dose": "Ketamine",
    }
    group_ylabels = {
        "any_opiate_dose": "Opiate dose (mcg)",
        "any_ketamine_dose": "Ketamine dose (mg)",
    }
    group_colors = {
        "any_opiate_dose": "#1f77b4",  # blue
        "any_ketamine_dose": "#E69F00",  # orange-yellow
    }

    for ax, group in zip(axes, group_order):
        group_df = plot_df[plot_df["medication_group"] == group]
        if group_df.empty:
            ax.set_visible(False)
            continue

        group_color = group_colors[group]
        sns.regplot(
            data=group_df,
            x=x_col,
            y="dose",
            ax=ax,
            color=group_color,
            scatter_kws={"alpha": 0.22, "s": 20, "edgecolor": "none", "zorder": 2},
            line_kws={"linewidth": 3.4, "alpha": 1.0, "zorder": 4},
            truncate=False,
        )

        ax.set_title(group_labels[group], fontsize=12, fontweight="semibold")
        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel(group_ylabels[group], fontsize=11)
        ax.tick_params(axis="both", labelsize=10)
        ax.grid(True, alpha=0.2)
        ax.spines[["top", "right"]].set_visible(False)

    if show_main_title:
        fig.suptitle(f"Medication dose vs {x_label.lower()} ({zero_text.lower()})", y=1.02, fontsize=12)

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to: {save_path}")

    plt.show()

# Single, publication-ready figure requested: physician age vs dose including zeros
figure_output_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/prehospital/analgesia/adult_trauma_analgesia/review2/physician_age_vs_medication_dose_including_zero.png"
plot_dose_vs_physician_metric(
    adult_df,
    x_col="physician_age",
    x_label="Physician age (years)",
    include_zero_dose=True,
    show_main_title=False,
    save_path=figure_output_path,
)

In [ ]:
# Spearman correlations (including zero doses), computed separately from plots
from scipy.stats import spearmanr

spearman_rows = []

for metric_col, metric_label in metric_columns:
    for dose_col in dose_columns:
        tmp = adult_df[[metric_col, dose_col]].replace([np.inf, -np.inf], np.nan).dropna()
        tmp = tmp[tmp[dose_col] >= 0]  # explicitly keep zeros

        if len(tmp) < 3:
            rho, p_value = np.nan, np.nan
        else:
            rho, p_value = spearmanr(tmp[metric_col], tmp[dose_col])

        spearman_rows.append({
            "predictor": metric_col,
            "predictor_label": metric_label,
            "dose_variable": dose_col,
            "n": len(tmp),
            "spearman_rho": rho,
            "p_value": p_value,
        })

spearman_include_zero_df = pd.DataFrame(spearman_rows)

print("Spearman correlations for medication dose vs physician metrics (including zero doses):")
for _, row in spearman_include_zero_df.iterrows():
    p_text = f"{row['p_value']:.4f}" if pd.notna(row['p_value']) else "nan"
    rho_text = f"{row['spearman_rho']:.3f}" if pd.notna(row['spearman_rho']) else "nan"
    print(
        f"- {row['dose_variable']} vs {row['predictor_label']}: "
        f"rho = {rho_text}, p = {p_text}, n = {int(row['n'])}"
    )

spearman_include_zero_df

In [ ]:
# Plot dose vs physician metric (excluding zero doses)
for metric_col, metric_label in metric_columns:
    plot_dose_vs_physician_metric(
        adult_df,
        x_col=metric_col,
        x_label=metric_label,
        include_zero_dose=False,
    )